# Network Expansion
## Model 3 (Test) - Stochastic Intertemporal Model

Scenario-based, multi-year formulation: investments (substations, lines, reinforcements) are shared across scenarios and decided once; operations are scenario-specific. Costs are discounted and budgets are enforced per year. Uncertainty enters via demand scenarios with probabilities.

### 1 - Imports

In [8]:
import numpy as np
import pandas as pd

from src.classes import DistributionNetwork, Substation
from src.solver import solve_network_stochastic

### 2 - Define the Distribution Network (shared)

In [9]:
# Nodes and loads
NODES = [f"N{i}" for i in range(1, 14)]
LOADS = [f"D{i}" for i in range(1, 11)]

# Initial substation
S1 = Substation("S1", "N4", 40, ['N3', 'N5', 'N9'], r_cost=200, edge_cost=50)
SUBSTATIONS = [S1]

line_cost = 50

base_load_capacity = {
    'D1': 6, 'D2': 3, 'D3': 2, 'D4': 5, 'D5': 3,
    'D6': 2, 'D7': 3, 'D8': 5, 'D9': 4, 'D10': 6
}

loads_locations = {
    'D1': 'N1', 'D2': 'N2', 'D3': 'N3', 'D4': 'N6', 'D5': 'N7',
    'D6': 'N8', 'D7': 'N9', 'D8': 'N11', 'D9': 'N12', 'D10': 'N13'
}

nodes_connected = {
    'N1': ['N2'],
    'N2': ['N1', 'N3'],
    'N3': ['N2', 'N4'],
    'N4': ['N3', 'N5', 'N9'],
    'N5': ['N4', 'N6'],
    'N6': ['N5','N7', 'N8'],
    'N7': ['N6'],
    'N8': ['N6'],
    'N9': ['N4', 'N10'],
    'N10': ['N9', 'N11', 'N13'],
    'N11': ['N10', 'N12'],
    'N12': ['N11'],
    'N13': ['N10']
}

DistributionNetwork = DistributionNetwork(
    NODES.copy(),
    LOADS.copy(),
    SUBSTATIONS.copy(),
    base_load_capacity.copy(),
    nodes_connected.copy(),
    loads_locations.copy(),
    line_cost
)

# Candidate substations
capacity = 15
s_cost = 100       # activation cost
l_cost = line_cost # feeder line cost
r_cost = 200       # capacity reinforcement cost

S2 = Substation("S2", "N14", capacity, ['N2'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S3 = Substation("S3", "N15", capacity, ['N6'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S4 = Substation("S4", "N16", capacity, ['N11', 'N13'], r_cost, edge_cost=l_cost, fix_cost=s_cost)

DistributionNetwork.add_candidate_substations([S2, S3, S4])

### 3 - Scenarios and horizon

In [10]:
years = list(range(1, 11))      # Years 1..10
R = 10                             # Capacity reinforcement size
dr = 0.05                          # Discount rate
# Dynamic budgets per year
B = {t: 180 + 25*(t-1) for t in years}

# Scenario growth rates (annual) and probabilities
scenarios_def = {
    'conservative': {'prob': 0.25, 'growth': 0.02},
    'base':         {'prob': 0.50, 'growth': 0.06},
    'high':         {'prob': 0.25, 'growth': 0.10},
}

# Build per-scenario, per-year demands
scenarios = {}
for name, data in scenarios_def.items():
    g = data['growth']
    scenarios[name] = {'prob': data['prob'], 'demands': {}}
    for t in years:
        factor = (1 + g) ** (t - 1)
        scenarios[name]['demands'][t] = {ld: val * factor for ld, val in base_load_capacity.items()}

# Quick demand check
total_demand = {name: {t: round(sum(scenarios[name]['demands'][t].values()), 2) for t in years} for name in scenarios}
total_demand

{'conservative': {1: 39.0,
  2: 39.78,
  3: 40.58,
  4: 41.39,
  5: 42.21,
  6: 43.06,
  7: 43.92,
  8: 44.8,
  9: 45.69,
  10: 46.61},
 'base': {1: 39.0,
  2: 41.34,
  3: 43.82,
  4: 46.45,
  5: 49.24,
  6: 52.19,
  7: 55.32,
  8: 58.64,
  9: 62.16,
  10: 65.89},
 'high': {1: 39.0,
  2: 42.9,
  3: 47.19,
  4: 51.91,
  5: 57.1,
  6: 62.81,
  7: 69.09,
  8: 76.0,
  9: 83.6,
  10: 91.96}}

### 4 - Solve stochastic model

In [11]:
solution = solve_network_stochastic(
    Network=DistributionNetwork,
    R=R,
    B=B,
    dr=dr,
    years=years,
    scenarios=scenarios,
    op_cost=2,
    OutputFlag=0
)

### 5 - Investment summary (scenario-invariant)

In [12]:
S_idx = list(range(1, len(DistributionNetwork.SUBSTATIONS)+1))
base_edge_cost = DistributionNetwork.edge_cost

# New lines built per year (costed edges only)
lines_added = {}
for t in years:
    added = set()
    for (i, j, s, tau), val in solution['b_on'].items():
        if tau == t and val > 0.5:
            e = (i, j)
            if base_edge_cost.get(e, 0) > 0:
                added.add(e)
    lines_added[t] = sorted(list(added))

# Expected demand and supply across scenarios
expected_demand = {}
expected_supply = {}
for t in years:
    exp_d = 0.0
    exp_s = 0.0
    for omega, data in scenarios.items():
        p = data['prob']
        exp_d += p * sum(data['demands'][t].values())
        exp_s += p * sum(solution['r'][(s, t, omega)] for s in S_idx)
    expected_demand[t] = round(exp_d, 2)
    expected_supply[t] = round(exp_s, 2)

investment_summary = pd.DataFrame({
    'Budget': [B[t] if not isinstance(B, dict) else B[t] for t in years],
    'System Demand (exp)': [expected_demand[t] for t in years],
    'System Supply (exp)': [expected_supply[t] for t in years],
    'System Capacity': [sum(solution['P'][(s, t)] for s in S_idx) for t in years],
    'Substations Active': [[s for s in S_idx if solution['w'][(s, t)] > 0.5] for t in years],
    'Lines Added': [lines_added[t] for t in years],
    'Cost substation (nom)': [round(solution['cost_components_nominal'][t]['substation'], 2) for t in years],
    'Cost lines (nom)': [round(solution['cost_components_nominal'][t]['lines'], 2) for t in years],
    'Cost reinforcement (nom)': [round(solution['cost_components_nominal'][t]['reinforcement'], 2) for t in years],
    'Cost (nominal)': [round(solution['cost_per_year_nominal'][t], 2) for t in years],
    'Cost (discounted)': [round(solution['cost_per_year'][t], 2) for t in years]
}, index=years)

investment_summary

,Budget,System Demand (exp),System Supply (exp),System Capacity,Substations Active,Lines Added,Cost substation (nom),Cost lines (nom),Cost reinforcement (nom),Cost (nominal),Cost (discounted)
1,180,39.00,39.00,40.0,[1],[],0.0,0.0,0.0,2.0,2.00
2,205,41.34,41.34,55.0,"[1, 2]","[(2, 14)]",100.0,50.0,0.0,154.0,146.67
3,230,43.85,43.85,55.0,"[1, 2]",[],0.0,0.0,0.0,4.0,3.63
4,255,46.55,46.55,55.0,"[1, 2]",[],0.0,0.0,0.0,4.0,3.46
5,280,49.45,49.45,70.0,"[1, 2, 4]","[(13, 16)]",100.0,50.0,0.0,156.0,128.34
6,305,52.56,52.56,70.0,"[1, 2, 4]",[],0.0,0.0,0.0,6.0,4.70
7,330,55.91,55.91,80.0,"[1, 2, 4]",[],0.0,0.0,200.0,206.0,153.72
8,355,59.52,59.52,105.0,"[1, 2, 3, 4]","[(6, 15)]",100.0,50.0,200.0,358.0,254.42
9,380,63.40,63.40,105.0,"[1, 2, 3, 4]",[],0.0,0.0,0.0,8.0,5.41
10,405,67.59,67.59,105.0,"[1, 2, 3, 4]",[],0.0,0.0,0.0,8.0,5.16


### 6 - Scenario operations

In [13]:
records = []
for omega, data in scenarios.items():
    for t in years:
        demand = sum(data['demands'][t].values())
        supply = sum(solution['r'][(s, t, omega)] for s in S_idx)
        active = [s for s in S_idx if solution['w'][(s, t)] > 0.5]
        lines_added = [ (i,j) for (i,j,s,tau), val in solution['b_on'].items() if tau == t and val > 0.5 and base_edge_cost.get((i,j),0) > 0 ]
        records.append({
            'Scenario': omega,
            'Prob': data['prob'],
            'Year': t,
            'Budget': B[t] if not isinstance(B, dict) else B[t],
            'Demand': round(demand, 2),
            'Supply': round(supply, 2),
            'System Capacity': sum(solution['P'][(s, t)] for s in S_idx),
            'Substations Active': active,
            'Lines Added': sorted(list(set(lines_added))),
            'Cost substation (nom)': round(solution['cost_components_nominal'][t]['substation'], 2),
            'Cost lines (nom)': round(solution['cost_components_nominal'][t]['lines'], 2),
            'Cost reinforcement (nom)': round(solution['cost_components_nominal'][t]['reinforcement'], 2),
            'Cost opex (nom)': round(solution['cost_components_nominal'][t]['opex'], 2),
            'Cost (nominal)': round(solution['cost_per_year_nominal'][t], 2),
            'Cost (discounted)': round(solution['cost_per_year'][t], 2)
        })

ops_summary = pd.DataFrame(records)
ops_summary

,Scenario,Prob,Year,Budget,Demand,Supply,System Capacity,Substations Active,Lines Added,Cost substation (nom),Cost lines (nom),Cost reinforcement (nom),Cost opex (nom),Cost (nominal),Cost (discounted)
0,conservative,0.25,1,180,39.00,39.00,40.0,[1],[],0.0,0.0,0.0,2.0,2.0,2.00
1,conservative,0.25,2,205,39.78,39.78,55.0,"[1, 2]","[(2, 14)]",100.0,50.0,0.0,4.0,154.0,146.67
2,conservative,0.25,3,230,40.58,40.58,55.0,"[1, 2]",[],0.0,0.0,0.0,4.0,4.0,3.63
3,conservative,0.25,4,255,41.39,41.39,55.0,"[1, 2]",[],0.0,0.0,0.0,4.0,4.0,3.46
4,conservative,0.25,5,280,42.21,42.21,70.0,"[1, 2, 4]","[(13, 16)]",100.0,50.0,0.0,6.0,156.0,128.34
5,conservative,0.25,6,305,43.06,43.06,70.0,"[1, 2, 4]",[],0.0,0.0,0.0,6.0,6.0,4.70
6,conservative,0.25,7,330,43.92,43.92,80.0,"[1, 2, 4]",[],0.0,0.0,200.0,6.0,206.0,153.72
7,conservative,0.25,8,355,44.80,44.80,105.0,"[1, 2, 3, 4]","[(6, 15)]",100.0,50.0,200.0,8.0,358.0,254.42
8,conservative,0.25,9,380,45.69,45.69,105.0,"[1, 2, 3, 4]",[],0.0,0.0,0.0,8.0,8.0,5.41
9,conservative,0.25,10,405,46.61,46.61,105.0,"[1, 2, 3, 4]",[],0.0,0.0,0.0,8.0,8.0,5.16


### 7 - Final-year assignments per scenario

In [14]:
last_year = years[-1]
S_idx = list(range(1, len(DistributionNetwork.SUBSTATIONS)+1))
N_idx = list(range(1, len(DistributionNetwork.NODES)+1))
for omega in scenarios:
    print(f"Assignments in Year {last_year} for scenario {omega}:")
    for n in N_idx:
        for s in S_idx:
            if solution['y'][(n, s, last_year, omega)] > 0.5:
                print(f"Node {DistributionNetwork.NODES[n-1]} assigned to {DistributionNetwork.SUBSTATIONS[s-1].id}")


Assignments in Year 10 for scenario conservative:
Node N1 assigned to S2
Node N2 assigned to S2
Node N3 assigned to S2
Node N4 assigned to S1
Node N5 assigned to S4
Node N6 assigned to S3
Node N7 assigned to S3
Node N8 assigned to S3
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S1
Node N12 assigned to S1
Node N13 assigned to S1
Node N14 assigned to S2
Node N15 assigned to S3
Node N16 assigned to S4
Assignments in Year 10 for scenario base:
Node N1 assigned to S2
Node N2 assigned to S2
Node N3 assigned to S1
Node N4 assigned to S1
Node N5 assigned to S4
Node N6 assigned to S3
Node N7 assigned to S3
Node N8 assigned to S3
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S1
Node N12 assigned to S1
Node N13 assigned to S1
Node N14 assigned to S2
Node N15 assigned to S3
Node N16 assigned to S4
Assignments in Year 10 for scenario high:
Node N1 assigned to S2
Node N2 assigned to S2
Node N3 assigned to S1
Node N4 assigned to S1
Node N5 assigned to S4
N